## 문장 분류 기법(Sentence Classification)

### 감정 분석(Sentiment Analysis)

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes trl gensim

허깅페이스 파이프라인 임포트

In [1]:
import transformers, torch
from transformers import pipeline

print(transformers.__version__)
print(torch.cuda.is_available())


Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu118 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0316 19:44:01.379000 36392 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


4.57.3
True


감성 분류 파이프라인 생성 (미세조정된 BERT 불러오기)

In [ ]:
clf = pipeline("sentiment-analysis")

# 문장 하나를 바로 분류
print(clf("The acting was great and the story was touching."))   # → POSITIVE
print(clf("The plot was boring and the pacing was slow."))       # → NEGATIVE

## 감정분석 파이프라인 과정
토크나이저-모델-후처리 

참고: https://huggingface.co/learn/llm-course/chapter2/2

In [1]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu118 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0319 17:14:43.763000 31400 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9598046541213989},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [2]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [6]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

outputs = model(**inputs, output_hidden_states=True)
print(outputs.hidden_states[-1].shape)
outputs.logits

torch.Size([2, 16, 768])


tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)

In [7]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

model.config.id2label

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


{0: 'NEGATIVE', 1: 'POSITIVE'}

### NER(Named-entity recognition) 처리

In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

nlp = pipeline("ner", model=model, tokenizer=tokenizer)
example = "My name is Wolfgang and I live in Berlin"

ner_results = nlp(example)
for entity in ner_results:
	print(f"Entity: {entity['word']}, Score: {entity['score']:.2f}, Label: {entity['entity']}")


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Entity: Wolfgang, Score: 1.00, Label: B-PER
Entity: Berlin, Score: 1.00, Label: B-LOC


In [4]:
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
print(example)
ner_results = nlp(example)
print(ner_results)
for entity in ner_results:
	print(f"Entity: {entity['word']}, Score: {entity['score']:.2f}, Label: {entity['entity']}")


My name is Sylvain and I work at Hugging Face in Brooklyn.
[{'entity': 'B-PER', 'score': np.float32(0.9986273), 'index': 4, 'word': 'S', 'start': 11, 'end': 12}, {'entity': 'B-PER', 'score': np.float32(0.93460387), 'index': 5, 'word': '##yl', 'start': 12, 'end': 14}, {'entity': 'B-PER', 'score': np.float32(0.791561), 'index': 6, 'word': '##va', 'start': 14, 'end': 16}, {'entity': 'B-PER', 'score': np.float32(0.90470797), 'index': 7, 'word': '##in', 'start': 16, 'end': 18}, {'entity': 'B-ORG', 'score': np.float32(0.9670037), 'index': 12, 'word': 'Hu', 'start': 33, 'end': 35}, {'entity': 'B-ORG', 'score': np.float32(0.8853459), 'index': 13, 'word': '##gging', 'start': 35, 'end': 40}, {'entity': 'I-ORG', 'score': np.float32(0.9884615), 'index': 14, 'word': 'Face', 'start': 41, 'end': 45}, {'entity': 'B-LOC', 'score': np.float32(0.9971419), 'index': 16, 'word': 'Brooklyn', 'start': 49, 'end': 57}]
Entity: S, Score: 1.00, Label: B-PER
Entity: ##yl, Score: 0.93, Label: B-PER
Entity: ##va, Sc

### 부록: Word2Vec 모델 사용법

In [5]:
import gensim, numpy as np
from gensim.models import Word2Vec

In [6]:
# Example text data
sentences = [
    "I feel good today",
    "The weather is clear and warm today",
    "I ate kimchi for lunch",
    "Exercising makes me feel better"
]

# Preprocess text data (split words by spaces)
sentences = [sentence.split() for sentence in sentences]

# Train Word2Vec model
model = Word2Vec(sentences, vector_size=10, window=3, min_count=1, sg=0)

# Check the vector for a specific word
vector = model.wv['feel']
print("Vector for the word 'feel':", vector)



Vector for the word 'feel': [ 0.07380505 -0.01533471 -0.04536613  0.06554051 -0.0486016  -0.01816018
  0.0287658   0.00991874 -0.08285215 -0.09448818]


In [8]:
# Find similar words
similar_words = model.wv.most_similar('feel', topn=3)
print("Words similar to 'feel':", similar_words)

Words similar to 'feel': [('today', 0.5436006188392639), ('warm', 0.35868826508522034), ('I', 0.32933861017227173)]
